# Semana 2 — Agentes Inteligentes e Busca Não-Informada
## INF0415 — Heurísticas e Modelagem Multiobjetivo
### Universidade Federal de Goiás — Bacharelado em Inteligência Artificial

**Objetivo:** Implementar BFS e DFS para o Quebra-Cabeça de 8 Peças, comparar experimentalmente completude, otimalidade, tempo e memória.

**Metodologia:** Pair programming — a cada 15-20 min, troquem quem codifica e quem revisa.

---

## 1. Formulação do Problema como Busca

Antes de programar, vamos formalizar o 8-puzzle usando o framework da aula:

| Componente | Definição |
|---|---|
| **Estado** | Tupla de 9 inteiros `(p0, p1, ..., p8)`, onde `0` = espaço vazio |
| **Estado inicial** | Qualquer permutação solucionável |
| **Estado objetivo** | `(1, 2, 3, 4, 5, 6, 7, 8, 0)` |
| **Sucessores** | Trocar o `0` com vizinho adjacente (cima, baixo, esquerda, direita) |
| **Custo** | 1 por movimento (uniforme) |
| **Espaço de estados** | 9!/2 = **181.440** estados alcançáveis |
| **Representação** | Permutação (discreto, determinístico) |

```
Exemplo: (1, 2, 3, 4, 0, 5, 6, 7, 8)

 1 | 2 | 3
-----------
 4 |   | 5
-----------
 6 | 7 | 8
```

> **Pergunta:** Qual tipo de espaço de busca é esse? Por que a árvore de busca pode ser *muito* maior que o grafo de estados?

## 2. Configuração e Imports

In [ ]:
from collections import deque
import time
import random

## 3. Classe `EightPuzzle` (fornecida)

**Leiam o código com atenção antes de prosseguir.** Entendam cada método.

In [ ]:
class EightPuzzle:
    """Representação do Quebra-Cabeça de 8 peças como problema de busca."""
    
    GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
    
    # Movimentos possíveis do espaço vazio
    MOVES = [
        ("↑", -1, 0),  # Cima
        ("↓",  1, 0),  # Baixo
        ("←",  0,-1),  # Esquerda
        ("→",  0, 1),  # Direita
    ]
    
    def __init__(self, state: tuple):
        assert len(state) == 9 and set(state) == set(range(9))
        self.state = tuple(state)
    
    def is_goal(self) -> bool:
        return self.state == self.GOAL
    
    def blank_pos(self) -> int:
        return self.state.index(0)
    
    def successors(self) -> list:
        """Gera todos os estados sucessores válidos.
        Retorna: lista de (ação: str, novo_estado: tuple)"""
        idx = self.blank_pos()
        row, col = idx // 3, idx % 3
        result = []
        for action, dr, dc in self.MOVES:
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_idx = new_row * 3 + new_col
                lst = list(self.state)
                lst[idx], lst[new_idx] = lst[new_idx], lst[idx]
                result.append((action, tuple(lst)))
        return result
    
    def display(self):
        """Imprime o tabuleiro 3x3 formatado."""
        s = self.state
        for i in range(3):
            row = [str(s[i*3+j]) if s[i*3+j] != 0 else ' ' for j in range(3)]
            print(" | ".join(row))
            if i < 2: print("-----------")
        print()

## 4. Geração de Estados Aleatórios Solucionáveis

Nem toda permutação do 8-puzzle é solucionável! Um estado é solucionável sse o número de **inversões** é **par**.

In [ ]:
def count_inversions(state: tuple) -> int:
    """Conta inversões (ignorando o 0)."""
    tiles = [t for t in state if t != 0]
    inv = 0
    for i in range(len(tiles)):
        for j in range(i + 1, len(tiles)):
            if tiles[i] > tiles[j]:
                inv += 1
    return inv

def is_solvable(state: tuple) -> bool:
    return count_inversions(state) % 2 == 0

def generate_random_state() -> tuple:
    """Gera estado aleatório garantidamente solucionável."""
    while True:
        state = list(range(9))
        random.shuffle(state)
        if is_solvable(tuple(state)):
            return tuple(state)

def generate_state_at_depth(depth: int) -> tuple:
    """Gera estado fazendo `depth` movimentos aleatórios a partir do objetivo."""
    state = EightPuzzle.GOAL
    visited = {state}
    for _ in range(depth):
        puzzle = EightPuzzle(state)
        succs = puzzle.successors()
        new_states = [s for _, s in succs if s not in visited]
        state = random.choice(new_states) if new_states else random.choice([s for _, s in succs])
        visited.add(state)
    return state

# Teste rápido
print("Estado objetivo:")
EightPuzzle(EightPuzzle.GOAL).display()

print("Estado aleatório (fácil, ~5 movimentos):")
easy = generate_state_at_depth(5)
EightPuzzle(easy).display()
print(f"Inversões: {count_inversions(easy)}, Solucionável: {is_solvable(easy)}")

## 5. Classe `Node` (fornecida)

Para reconstruir o caminho da solução, rastreamos **pai** e **ação** de cada nó.

In [ ]:
class Node:
    def __init__(self, state, parent=None, action=None, depth=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.depth = depth
    
    def path(self):
        """Lista de ações do início até este nó."""
        actions = []
        node = self
        while node.parent is not None:
            actions.append(node.action)
            node = node.parent
        actions.reverse()
        return actions
    
    def path_states(self):
        """Sequência de estados do início até este nó."""
        states = []
        node = self
        while node is not None:
            states.append(node.state)
            node = node.parent
        states.reverse()
        return states

## 6. BFS — Busca em Largura 🔧

### ✏️ IMPLEMENTEM AQUI

**Algoritmo:**
1. Fronteira = **fila FIFO** (`deque`)
2. Visitados = `set`
3. Inserir nó raiz na fronteira e nos visitados
4. Loop: retirar da **esquerda** (`popleft`), testar objetivo, expandir sucessores

**Dica:** `node.state` é tupla → pode ser chave do `set`.

In [ ]:
def bfs(initial_state: tuple) -> dict:
    """Busca em Largura para o 8-puzzle."""
    start_time = time.time()
    root = Node(initial_state)
    
    if EightPuzzle(initial_state).is_goal():
        return {"solution": [], "nodes_expanded": 0,
                "depth": 0, "explored_size": 1,
                "time": time.time() - start_time}
    
    # ============================================
    # 👇 IMPLEMENTEM A PARTIR DAQUI 👇
    # ============================================
    
    frontier = deque()          # Fila FIFO
    explored = set()            # Estados visitados
    nodes_expanded = 0
    
    # TODO: Adicionar nó raiz à fronteira e ao explored
    
    
    # TODO: Loop principal
    # while frontier:
    #     node = frontier.popleft()
    #     nodes_expanded += 1
    #     for action, new_state in EightPuzzle(node.state).successors():
    #         if new_state not in explored:
    #             child = Node(new_state, parent=node,
    #                          action=action, depth=node.depth+1)
    #             if EightPuzzle(new_state).is_goal():
    #                 return {"solution": child.path(),
    #                         "nodes_expanded": nodes_expanded,
    #                         "depth": child.depth,
    #                         "explored_size": len(explored),
    #                         "time": time.time() - start_time}
    #             frontier.append(child)
    #             explored.add(new_state)
    
    
    return {"solution": None, "nodes_expanded": nodes_expanded,
            "depth": -1, "explored_size": len(explored),
            "time": time.time() - start_time}

### Teste do BFS
Se correto, o estado fácil deve ser resolvido em <1s.

In [ ]:
test_easy = generate_state_at_depth(5)
print("Estado inicial:")
EightPuzzle(test_easy).display()

result = bfs(test_easy)
if result["solution"] is not None:
    print(f"✅ Solução encontrada!")
    print(f"   Ações: {' → '.join(result['solution'])}")
    print(f"   Profundidade: {result['depth']}")
    print(f"   Nós expandidos: {result['nodes_expanded']:,}")
    print(f"   Tempo: {result['time']:.4f}s")
else:
    print("❌ Solução não encontrada — revise a implementação!")

## 7. DFS — Busca em Profundidade 🔧

### ✏️ IMPLEMENTEM AQUI

**Diferenças vs. BFS:**
- Fronteira = **pilha LIFO** (lista + `pop()`)
- Incluir `max_depth` para evitar loops infinitos
- DFS **não** garante solução ótima

**Dica:** Trocar `deque` + `popleft()` por `list` + `pop()` e checar profundidade.

In [ ]:
def dfs(initial_state: tuple, max_depth: int = 50) -> dict:
    """Busca em Profundidade para o 8-puzzle."""
    start_time = time.time()
    root = Node(initial_state)
    
    if EightPuzzle(initial_state).is_goal():
        return {"solution": [], "nodes_expanded": 0,
                "depth": 0, "explored_size": 1,
                "time": time.time() - start_time}
    
    # ============================================
    # 👇 IMPLEMENTEM A PARTIR DAQUI 👇
    # ============================================
    
    frontier = []               # Pilha LIFO
    explored = set()
    nodes_expanded = 0
    
    # TODO: Adicionar nó raiz à fronteira e ao explored
    
    
    # TODO: Loop principal (similar ao BFS, mas...)
    # while frontier:
    #     node = frontier.pop()    # <-- LIFO!
    #     nodes_expanded += 1
    #     if node.depth < max_depth:  # <-- Limite!
    #         for action, new_state in EightPuzzle(node.state).successors():
    #             if new_state not in explored:
    #                 child = Node(new_state, parent=node,
    #                              action=action, depth=node.depth+1)
    #                 if EightPuzzle(new_state).is_goal():
    #                     return {"solution": child.path(),
    #                             "nodes_expanded": nodes_expanded,
    #                             "depth": child.depth,
    #                             "explored_size": len(explored),
    #                             "time": time.time() - start_time}
    #                 frontier.append(child)
    #                 explored.add(new_state)
    
    
    return {"solution": None, "nodes_expanded": nodes_expanded,
            "depth": -1, "explored_size": len(explored),
            "time": time.time() - start_time}

### Teste do DFS

In [ ]:
print("Estado inicial:")
EightPuzzle(test_easy).display()

result_dfs = dfs(test_easy, max_depth=50)
if result_dfs["solution"] is not None:
    print(f"✅ Solução encontrada! Prof: {result_dfs['depth']}")
    print(f"   Nós expandidos: {result_dfs['nodes_expanded']:,}")
    print(f"   Tempo: {result_dfs['time']:.4f}s")
else:
    print("❌ Solução não encontrada")

## 8. Visualização da Solução

Função para ver passo a passo a solução encontrada.

In [ ]:
def visualize_solution(initial_state, actions):
    """Mostra passo a passo a solução no tabuleiro."""
    state = initial_state
    print("=== Estado Inicial ===")
    EightPuzzle(state).display()
    
    for i, action in enumerate(actions):
        puzzle = EightPuzzle(state)
        for a, new_state in puzzle.successors():
            if a == action:
                state = new_state
                break
        print(f"--- Passo {i+1}: mover vazio {action} ---")
        EightPuzzle(state).display()
    
    print(f"✅ Objetivo alcançado em {len(actions)} movimentos!")

# Visualizar solução do BFS
r = bfs(test_easy)
if r["solution"]:
    visualize_solution(test_easy, r["solution"])

## 9. Experimento Comparativo 🧪

### ✏️ Rodem e analisem os resultados

Compara BFS e DFS em 5 estados com dificuldade crescente.

In [ ]:
difficulties = [
    ("Fácil",     5),
    ("Fácil+",   10),
    ("Médio",    15),
    ("Difícil",   20),
    ("Aleatório", None),
]

test_states = []
for label, depth in difficulties:
    state = generate_state_at_depth(depth) if depth else generate_random_state()
    test_states.append((label, state))

print(f"{"Estado":<12} {"Alg.":<5} {"Nós Exp.":>10} {"Prof.":>6} "
      f"{"Tempo(s)":>10} {"Visitados":>10} {"Ótima?":>7}")
print("=" * 72)

for label, state in test_states:
    r_bfs = bfs(state)
    bd = r_bfs["depth"] if r_bfs["solution"] is not None else "FALHA"
    print(f"{label:<12} {"BFS":<5} {r_bfs["nodes_expanded"]:>10,} {str(bd):>6} "
          f"{r_bfs["time"]:>10.4f} {r_bfs["explored_size"]:>10,} {"—":>7}")
    
    r_dfs = dfs(state, max_depth=50)
    dd = r_dfs["depth"] if r_dfs["solution"] is not None else "FALHA"
    opt = ""
    if r_dfs["solution"] is not None and r_bfs["solution"] is not None:
        opt = "Sim" if r_dfs["depth"] == r_bfs["depth"] else "Não"
    print(f"{"":<12} {"DFS":<5} {r_dfs["nodes_expanded"]:>10,} {str(dd):>6} "
          f"{r_dfs["time"]:>10.4f} {r_dfs["explored_size"]:>10,} {opt:>7}")
    print("-" * 72)

## 10. Perguntas para Discussão ✍️

Respondam **em dupla** (cliquem duas vezes para editar):

### P1. BFS sempre encontrou a solução mais curta? Por quê?
*Resposta:*

### P2. DFS encontrou solução em todos os casos? A solução foi ótima?
*Resposta:*

### P3. Qual algoritmo usou mais memória (|Visited|)? A diferença foi consistente?
*Resposta:*

### P4. O que aconteceria sem limite de profundidade no DFS?
*Resposta:*

### P5. Para estados difíceis, BFS ficou lento ou consumiu muita memória?
*Resposta:*

### P6. Se só pudesse usar UM algoritmo para o VRP (centenas de cidades), qual e por quê?
*Resposta:*

### P7. O que precisaríamos para fazer *melhor* que ambos? (Tease Semana 3!)
*Resposta:*

## 🏆 Desafio Bônus: Profundidade Iterativa (IDS)

Combina **memória do DFS** com **completude do BFS**.

Roda DFS com limite L=0, 1, 2, ... até encontrar solução.

### ✏️ Implementem se sobrar tempo (ou como tarefa para casa)

In [ ]:
def ids(initial_state: tuple, max_limit: int = 80) -> dict:
    """Busca de Profundidade Iterativa."""
    start_time = time.time()
    total_expanded = 0
    
    for limit in range(max_limit + 1):
        result = dfs(initial_state, max_depth=limit)
        total_expanded += result["nodes_expanded"]
        if result["solution"] is not None:
            return {"solution": result["solution"],
                    "nodes_expanded": total_expanded,
                    "depth": result["depth"],
                    "explored_size": result["explored_size"],
                    "time": time.time() - start_time}
    
    return {"solution": None, "nodes_expanded": total_expanded,
            "depth": -1, "explored_size": 0,
            "time": time.time() - start_time}

# Comparar BFS vs IDS
test_ids = generate_state_at_depth(15)
print("Comparando BFS vs IDS:")
rb = bfs(test_ids)
ri = ids(test_ids)
print(f"BFS: {rb["nodes_expanded"]:>8,} nós, prof={rb["depth"]}, tempo={rb["time"]:.3f}s")
if ri["solution"]:
    print(f"IDS: {ri["nodes_expanded"]:>8,} nós, prof={ri["depth"]}, tempo={ri["time"]:.3f}s")
    ratio = ri["nodes_expanded"] / max(rb["nodes_expanded"], 1)
    print(f"\nRatio IDS/BFS: {ratio:.2f}x nós expandidos")
    print(f"Mesma profundidade? {ri["depth"] == rb["depth"]}")

---
## 📚 Para a Semana 3

Na próxima aula, vamos adicionar uma **heurística** (distância Manhattan) a este MESMO código, transformando BFS em **A***.

**Leitura:** Russell & Norvig, 4ª Ed., Cap. 3.5–3.6 (Busca Informada e A*).

**Mantenham este notebook organizado** — vamos reutilizá-lo!

---
*INF0415 — Heurísticas e Modelagem Multiobjetivo — UFG 2026/2*